# 09.1 — Environment detection

A production pipeline doesn't run in one place. You develop it on a **laptop**, test it on a **lab server**, and run the real sweeps on an **HPC cluster** — and each of those has a completely different filesystem layout. Raw data lives at `/Volumes/...` on the laptop but `/home/...` on the cluster; scratch space is `/tmp` here but a fast parallel filesystem there. Hardcoding paths would make the code run in exactly one environment. Instead, the pipeline **detects which environment it's in at runtime** and resolves the right base directories — so the *same code* runs everywhere. This notebook is that detector.

This notebook covers:

* Why a deployable pipeline must detect its environment.
* The three environments (Local / TEBA / ACCRE) and their path layouts.
* `get_base_paths` and auto-detection (the ACCRE-mount check).
* The three path roles: input, output, temporary.

**Prerequisites:** [00.5 the command line](../00_orientation/00.5_the_command_line_for_matlab_users.ipynb), [08.1 folder hierarchy](../08_output_and_analysis/08.1_folder_hierarchy_generation.ipynb).

## Section 1 — What MATLAB does

`cgg_getBaseFolders` detects the host and returns the base directories; `cgg_checkACCREMounted` decides whether the cluster filesystem is reachable:

```matlab
function [inputDir, outputDir, tempDir] = cgg_getBaseFolders()
    if cgg_checkACCREMounted()          % is the cluster data mount visible?
        inputDir = '/home/...';          % ACCRE layout
    elseif isLocal()
        inputDir = '/Volumes/...';       % laptop layout
    end
    % ... returns the right roots for THIS machine
end
```

Every script that reads data or writes results calls this first, so no script hardcodes a path. The port maps it to `get_base_paths()` returning a typed `BasePaths`, with `detect_environment()` doing the host detection (mirroring `cgg_checkACCREMounted`).

## Section 2 — The Python concepts you need

### 2.1 — Why detect the environment

Hardcoding `/Users/me/data/...` in your code means it runs on *your* machine and nowhere else. The moment you `ssh` to the cluster, every path is wrong. Environment detection solves this by adding one layer of indirection: the code asks "where am I?" and gets back the correct base directories for *this* host. Write `paths.input / "session.mat"` instead of `/Users/me/data/session.mat`, and the same line resolves correctly on the laptop, the lab server, and the cluster. It's the deployment version of the config-in-the-path idea ([08.1](../08_output_and_analysis/08.1_folder_hierarchy_generation.ipynb)): don't bake host specifics into code.

### 2.2 — The three environments

In [ ]:
from neural_data_decoding.utils.paths import get_base_paths, RuntimeEnvironment

for env in RuntimeEnvironment:
    p = get_base_paths(environment=env)      # force each environment to compare
    print(f"{env.value.upper():6}")
    print(f"   input     (read-only data): {p.input}")
    print(f"   output    (preserved results): {p.output}")
    print(f"   temporary (scratch): {p.temporary}")
    print()

Three environments, three layouts:

* **LOCAL** — a personal workstation (this laptop). Data and results under the user's home / an ACCRE-mount shortcut; scratch in a local data dir.
* **TEBA** — the lab server, with data under `/data`.
* **ACCRE** — the HPC cluster: read-only data under `/home`, but scratch under a fast lab-scratch filesystem (`/data/womelsdorf_lab/...`).

The *paths differ per host*, but the *roles* (input/output/temporary) are the same everywhere — which is exactly what lets one code path serve all three. (The paths shown are host-specific and include the current user; on your machine they'd resolve to your own directories.)

### 2.3 — Auto-detection

In [ ]:
from neural_data_decoding.utils.paths import detect_environment

detected = detect_environment()
print("auto-detected environment on this host:", detected.value)
paths = get_base_paths()                     # no argument → auto-detect
print("resolved input root:", paths.input)
print("resolved environment:", paths.environment.value)
print("→ get_base_paths() with no argument detects the host and returns its layout.")

`detect_environment()` inspects the host — chiefly *whether the ACCRE cluster data mount is visible* (`_accre_mount_visible`, mirroring `cgg_checkACCREMounted.m`) plus other host signals — and picks the matching `RuntimeEnvironment`. `get_base_paths()` with no argument calls it automatically, so production code never needs to know *which* environment it's in; it just asks for `paths.input` and gets the right root. On this machine it detects `local`; submitted to the cluster, the same call would detect `accre` and return the cluster layout.

### 2.4 — The typed API (enum + dataclass)

MATLAB's version returns three bare strings — easy to mix up (which one was output?). The port uses a **typed** API: a `RuntimeEnvironment` *enum* (not a magic string) and a `BasePaths` *dataclass* with named fields. You can't accidentally swap `input` and `output`, and the environment is a checked enum value, not a typo-prone string ([01.4 classes](../01_python_for_matlab_users/01.4_classes_and_oop.ipynb)). You can also *force* an environment for testing — `get_base_paths(environment=RuntimeEnvironment.ACCRE)` — to check cluster-path logic from your laptop, which is how the tests verify all three layouts without three machines.

In [ ]:
# Forcing an environment lets you test cluster logic locally:
accre = get_base_paths(environment=RuntimeEnvironment.ACCRE)
print("forced ACCRE input starts with /home/ ?", str(accre.input).startswith("/home/"))
print("forced ACCRE scratch is the lab filesystem?", "womelsdorf_lab" in str(accre.temporary))
print("→ the enum + dataclass make each environment's layout testable from anywhere.")

### 2.5 — Three path roles: input, output, temporary

The split into `input` / `output` / `temporary` isn't arbitrary — the three have different *storage semantics*, especially on a cluster:

* **`input`** — read-only raw/preprocessed data. You never write here; it's shared, backed up, and often mounted read-only.
* **`output`** — final results to *preserve* (the `CM_Table.mat` files, [08.2](../08_output_and_analysis/08.2_writing_mat_files_for_matlab.ipynb)). Backed up, kept after the job ends.
* **`temporary`** — scratch: checkpoints, intermediate artifacts, sweep working files. On a cluster this is a *fast but ephemeral* parallel filesystem — huge and quick, but periodically purged. You write hot data here during the run, then copy the keepers to `output`.

Putting checkpoints on `temporary` (fast, purgeable) and final tables on `output` (preserved) matches how HPC storage is actually tiered. Conflating them — writing results to scratch — risks losing them to a purge; writing scratch to the backed-up volume wastes precious quota. The three-way split encodes that discipline.

## Section 3 — The `neural_data_decoding` implementation

`get_base_paths` and the `BasePaths` it returns:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/utils/paths.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("class BasePaths"))
j0 = next(j for j in range(i, len(src)) if src[j].strip() == "input: Path")
for k in range(j0, j0 + 4):
    print(f"{k + 1:4} | {src[k]}")
i2 = next(j for j, line in enumerate(src) if line.startswith("def get_base_paths"))
for k in range(i2, i2 + 5):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `BasePaths` is a dataclass with four fields: `input`, `output`, `temporary` (all `Path`), plus `environment` (the `RuntimeEnvironment` they belong to) — named, typed, unmixable (§2.4).
* `get_base_paths(*, environment=None, want_teba=False)` — `environment=None` auto-detects; passing one forces it. `want_teba` picks TEBA-style external-volume paths when local (a laptop-with-external-drive case).
* `detect_environment()` and `_accre_mount_visible()` port `cgg_getBaseFolders.m` + `cgg_checkACCREMounted.m` — the host detection is where the environment-specific logic lives.
* `RuntimeEnvironment` is a `str`-backed `Enum` (`LOCAL`/`TEBA`/`ACCRE`), so it compares cleanly and serializes to a plain string in configs, while still being a checked type in code.
* Every path-consuming site (the data loader, the result writer, [08.1](../08_output_and_analysis/08.1_folder_hierarchy_generation.ipynb)) takes its root from here — so porting to a new environment is *one* change to this module, not a hunt through the codebase.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the same code, three layouts

Build a data path (`input / "session_0001.mat"`) under each environment and show the *code* is identical while the *resolved path* differs.

In [ ]:
# Reveal:
for env in RuntimeEnvironment:
    p = get_base_paths(environment=env)
    data_file = p.input / "session_0001.mat"      # IDENTICAL code for every environment
    print(f"{env.value:6}: {data_file}")
print("→ one line of code, three correct paths — that's what environment detection buys.")

### Exercise 4.2 — scratch vs preserved

Show that on ACCRE the `temporary` (scratch) root differs from `output` (preserved), so checkpoints and final results land on different storage tiers.

In [ ]:
# Reveal:
accre = get_base_paths(environment=RuntimeEnvironment.ACCRE)
print("ACCRE output   (preserved):", accre.output)
print("ACCRE temporary (scratch): ", accre.temporary)
print("different filesystems?", accre.output.parts[:3] != accre.temporary.parts[:3])
print("→ checkpoints → fast scratch; final tables → preserved output. Two tiers, by design.")

## Section 5 — Common errors

### `FileNotFoundError` on the cluster but not locally

A hardcoded path slipped in somewhere (§2.1). Route *every* path through `get_base_paths()`; grep the codebase for absolute paths that bypass it.

### Detected the wrong environment

`detect_environment()` keys off host signals (the ACCRE-mount check, §2.3). If it guesses wrong — e.g., an external drive that looks like the cluster mount — force it explicitly: `get_base_paths(environment=...)` or the `want_teba` flag.

### Results lost after the cluster job ends

Final results were written to `temporary` (scratch), which gets purged (§2.5). Write preserved outputs to `output`; use `temporary` only for checkpoints and intermediates you can regenerate.

### Scratch quota exhausted / slow I/O

Hot data (checkpoints, augmentation caches) belongs on `temporary` (the fast parallel filesystem), not `output`. Writing high-frequency I/O to the backed-up `output` volume is slow and burns quota.

### Tests need three machines

They don't — force each environment with the `environment=` argument (§2.4). The typed API makes every layout reproducible from one host, which is how the path tests cover all three.

## Section 6 — Further reading

- [`src/neural_data_decoding/utils/paths.py`](../../src/neural_data_decoding/utils/paths.py) — `get_base_paths`, `detect_environment`, `RuntimeEnvironment` (ports `cgg_getBaseFolders.m`).
- [12-factor config](https://12factor.net/config) — the general principle of not baking environment specifics into code.
- [08.1 folder hierarchy](../08_output_and_analysis/08.1_folder_hierarchy_generation.ipynb) — what gets built *under* these base paths.

Next up: **[09.2 SLURM dispatch](09.2_slurm_dispatch.ipynb)** — turning one config into hundreds of cluster jobs.